# Check a Kaggle dataset before training on it

Three cells. No GPU needed, nothing is modified.

Run this before the training notebook. It answers five questions a Kaggle page
cannot.

1. What are the real folder names?
2. How many **genuine photographs**, as opposed to image files? Many datasets are
   padded with rotated and flipped copies saved as separate files.
3. Are there byte-identical duplicates, including one image sitting in two classes?
4. Are the classes balanced?
5. Does the dataset ship both an original and a pre-augmented collection?

This matters because the dataset in the previous submission held 2080 files built
from only 160 photographs, with 111 exact duplicates. Checking first stops that
repeating.

---
## Cell 1 - Install, import, sign in

Pick your `kaggle.json` when the button appears. Get it from kaggle.com ->
profile picture -> Settings -> API Tokens tab -> **Create Legacy API Key**.

In [ ]:
# ========================= INSTALL =========================
!pip install -q kaggle

# ========================= IMPORTS =========================
import os, re, json, hashlib, shutil
from collections import defaultdict
from PIL import Image
from google.colab import files

# ========================= SETTINGS =========================
WORK = "candidates"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Folders of processed copies rather than real photographs.
SKIP_WORDS = ("augment", "hog", "black and white", "grayscale", "greyscale")

# Filename fragments marking an augmented copy of another photograph.
AUG_HINTS = ("rotation", "rotate", "zoom", "flip", "mirror", "contrast",
             "constract", "crop", "translation", "rotozoom", "pil_",
             "bright", "aug", "noise", "shear")

os.makedirs(WORK, exist_ok=True)

# ========================= CREDENTIALS =========================
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Choose your kaggle.json file:")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(up["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Signed in as:", json.load(open("/root/.kaggle/kaggle.json"))["username"])

---
## Cell 2 - List your candidates and download them

Search kaggle.com for `cotton leaf disease`, `cotton pest`, or
`cotton disease detection`. Open anything promising and check the **Data** tab.

To get a slug reliably, click the **three dots** next to the Download button and
choose **Copy API command**. The part after `-d` is the slug. Do not click the blue
Download button, because that pulls the whole dataset onto your laptop instead of
straight to Colab.

Two or three candidates is plenty.

In [ ]:
CANDIDATES = [
    "PASTE/FIRST-SLUG",
    # "PASTE/SECOND-SLUG",
    # "PASTE/THIRD-SLUG",
]

for slug in CANDIDATES:
    dest = os.path.join(WORK, slug.replace("/", "__"))
    if os.path.exists(dest) and os.listdir(dest):
        print("already downloaded:", slug)
        continue
    os.makedirs(dest, exist_ok=True)
    print("\n=== downloading", slug, "===")
    rc = os.system("kaggle datasets download -d " + slug + " -p " + dest + " --unzip -q")
    if rc != 0:
        print("FAILED. 403 = open the dataset page in a browser and accept its "
              "terms. 404 = the slug is wrong.")
print("\ndone")

---
## Cell 3 - Audit and compare

A report card per candidate, then a side-by-side table.

**How to choose, in order of importance:**

1. **Reject anything with duplicates across classes.** One photograph carrying two
   labels teaches the model contradictions and corrupts the test set.
2. **Compare the photographs column, never the files column.** 5000 files from 300
   photographs is a smaller dataset than 900 files from 900 photographs.
3. **Check the class names are ones you can defend.** If they differ from your
   current four, that is allowed, but the title, aim and objectives must be
   rewritten to name the classes you actually detect. A marker already criticised
   the title for not matching the work.
4. **Watch imbalance.** Above about 3x, plain accuracy misleads, because a model
   can score well by favouring the largest class. Report macro F1.
5. **Do not merge two datasets.** Kaggle uploads often repackage the same source
   images, which puts one photograph on both sides of your split.

In [ ]:
def base_id(fname):
    stem = os.path.splitext(fname)[0]
    if any(h in stem.lower() for h in AUG_HINTS):
        m = re.search(r"(\d+)$", stem)
        if m:
            return m.group(1)
    return stem


def find_class_levels(root):
    """Every folder whose subfolders hold images, deepest kept."""
    found = []
    for dirpath, dirnames, _ in os.walk(root):
        if not dirnames:
            continue
        counts = []
        for d in dirnames:
            p = os.path.join(dirpath, d)
            try:
                counts.append(sum(1 for f in os.listdir(p)
                                  if os.path.isfile(os.path.join(p, f))
                                  and f.lower().endswith(IMG_EXT)))
            except OSError:
                counts.append(0)
        if counts and min(counts) > 0 and sum(counts) > 20:
            found.append((sum(counts), dirpath))
    keep = []
    for total, path in found:
        deeper = any(p.startswith(path + os.sep) for _, p in found if p != path)
        if not deeper:
            keep.append((total, path))
    return keep


def pick_level(root):
    levels = find_class_levels(root)
    if not levels:
        return None
    clean = [l for l in levels if not any(w in l[1].lower() for w in SKIP_WORDS)]
    pool = clean or levels
    preferred = [l for l in pool if "original" in l[1].lower()]
    pool = preferred or pool
    pool.sort(key=lambda t: -t[0])
    return pool[0][1]


def audit(slug):
    root = os.path.join(WORK, slug.replace("/", "__"))
    print("\n" + "=" * 68)
    print("DATASET:", slug)
    print("=" * 68)
    if not os.path.isdir(root) or not os.listdir(root):
        print("  nothing downloaded")
        return None

    all_levels = find_class_levels(root)
    if len(all_levels) > 1:
        print("  this dataset holds more than one image collection:")
        for total, path in sorted(all_levels, key=lambda t: -t[0]):
            tag = "   <-- processed copies, not real photographs" \
                  if any(w in path.lower() for w in SKIP_WORDS) else ""
            print("     {}  [{} images]{}".format(
                os.path.relpath(path, root), total, tag))
        print("  the original collection is audited below.\n")

    level = pick_level(root)
    if level is None:
        print("  could not find class folders. Structure:")
        for dp, dn, fn in os.walk(root):
            depth = dp.replace(root, "").count(os.sep)
            if depth > 3:
                continue
            n = sum(1 for f in fn if f.lower().endswith(IMG_EXT))
            print("   " + "  " * depth + (os.path.basename(dp) or ".") +
                  ("  [{} images]".format(n) if n else ""))
        return None

    print("  class folders:", os.path.relpath(level, root))
    classes = sorted(d for d in os.listdir(level)
                     if os.path.isdir(os.path.join(level, d)))

    rows, hashes, sizes = [], defaultdict(list), []
    total_files = total_photos = 0
    for cls in classes:
        cdir = os.path.join(level, cls)
        fnames = [f for f in os.listdir(cdir) if f.lower().endswith(IMG_EXT)]
        groups = defaultdict(list)
        for f in fnames:
            groups[base_id(f)].append(f)
            try:
                h = hashlib.md5(open(os.path.join(cdir, f), "rb").read()).hexdigest()
                hashes[h].append(cls + "/" + f)
            except OSError:
                pass
        for f in fnames[:5]:
            try:
                with Image.open(os.path.join(cdir, f)) as im:
                    sizes.append(im.size)
            except Exception:
                pass
        rows.append((cls, len(fnames), len(groups)))
        total_files += len(fnames)
        total_photos += len(groups)

    print("\n  {:30s} {:>8s} {:>13s}".format("class", "files", "photographs"))
    print("  " + "-" * 54)
    for cls, nf, ng in rows:
        print("  {:30s} {:8d} {:13d}".format(cls[:30], nf, ng))
    print("  " + "-" * 54)
    print("  {:30s} {:8d} {:13d}".format("TOTAL", total_files, total_photos))

    dups = {h: v for h, v in hashes.items() if len(v) > 1}
    redundant = sum(len(v) - 1 for v in dups.values())
    cross = [v for v in dups.values() if len(set(x.split("/")[0] for x in v)) > 1]
    print("\n  identical duplicate files :", redundant)
    print("  duplicates across classes :", len(cross),
          "  <-- serious, one image labelled two ways" if cross else "")
    for grp in cross[:2]:
        extra = len(grp) - 4
        print("     ", ", ".join(grp[:4]),
              ("... and " + str(extra) + " more") if extra > 0 else "")

    counts = [ng for _, _, ng in rows]
    imbalance = max(counts) / min(counts) if counts and min(counts) else float("inf")
    if sizes:
        ws = sorted(w for w, h in sizes)
        hs = sorted(h for w, h in sizes)
        print("\n  image size sample: {}x{} to {}x{}".format(ws[0], hs[0], ws[-1], hs[-1]))
    print("  class imbalance (largest / smallest): {:.1f}x".format(imbalance))

    ratio = total_files / max(1, total_photos)
    print("\n  VERDICT")
    if ratio > 1.5:
        print("    Padded with augmented copies: about {:.0f} files per photograph.".format(ratio))
        print("    Real size is {} photographs, not {} images.".format(total_photos, total_files))
        print("    Usable, but the split must group by photograph. The training")
        print("    notebook already does that.")
    else:
        print("    No offline augmentation. Each file is its own photograph.")
    if total_photos < 400:
        print("    SMALL: {} photographs. Workable with transfer learning, but".format(total_photos))
        print("    state it as a limitation.")
    elif total_photos < 1500:
        print("    REASONABLE: {} photographs.".format(total_photos))
    else:
        print("    GOOD: {} photographs.".format(total_photos))
    if cross:
        print("    AVOID unless you remove the cross-class duplicates first.")
    if imbalance > 3:
        print("    Imbalanced at {:.1f}x. Use class weights and report macro F1.".format(imbalance))

    return dict(slug=slug, classes=classes, files=total_files,
                photographs=total_photos, duplicates=redundant,
                cross_class=len(cross), imbalance=imbalance)


reports = [r for r in (audit(s) for s in CANDIDATES) if r]

if reports:
    print("\n\n" + "=" * 78)
    print("SIDE BY SIDE")
    print("=" * 78)
    print("{:38s} {:>7s} {:>7s} {:>8s} {:>6s} {:>7s}".format(
        "dataset", "photos", "files", "classes", "dupes", "imbal"))
    print("-" * 78)
    for r in sorted(reports, key=lambda r: -r["photographs"]):
        print("{:38s} {:7d} {:7d} {:8d} {:6d} {:6.1f}x".format(
            r["slug"][:38], r["photographs"], r["files"],
            len(r["classes"]), r["duplicates"], r["imbalance"]))
    print("\nCLASS NAMES")
    for r in reports:
        print("\n  " + r["slug"])
        for c in r["classes"]:
            print("     ", c)
    print("\nCopy the winning slug and its class names into Cell 1 of "
          "TRAIN_IN_COLAB.ipynb.")